# Australian Graduate Employment Outcomes Analysis

This notebook is the Python version of the QILT GOS-L graduate employment analysis portfolio project.

**Focus:** Computing and Information Systems (CIS) graduates in Australia  
**Data source:** QILT Graduate Outcomes Survey - Longitudinal (GOS-L) 2025 public summary tables

The notebook demonstrates a data analyst workflow using Python: reading cleaned CSV data, calculating KPIs, creating charts, and summarising insights.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

## 2. Load Data

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

area_path = ROOT / "data/processed/qilt_gosl_2025_area_features.csv"
institution_path = ROOT / "outputs/tables/institutional_summary.csv"

area = pd.read_csv(area_path)
institutions = pd.read_csv(institution_path)

area.head()

In [ ]:
area.info()

## 3. CIS KPI Summary

In [ ]:
cis = area.loc[area["is_cis"] == "Yes"].iloc[0]

cis_kpis = pd.DataFrame(
    {
        "Metric": [
            "Medium-term full-time employment rate",
            "Medium-term overall employment rate",
            "Short-term median salary",
            "Medium-term median salary",
            "Salary growth",
            "Male medium-term median salary",
            "Female medium-term median salary",
            "Gender pay gap",
        ],
        "Result": [
            f"{cis['medium_term_full_time_employment_rate']:.1f}%",
            f"{cis['medium_term_overall_employment_rate']:.1f}%",
            f"AUD {cis['short_term_median_salary_aud']:,.0f}",
            f"AUD {cis['medium_term_median_salary_aud']:,.0f}",
            f"AUD {cis['salary_growth']:,.0f}",
            f"AUD {cis['male_medium_salary_aud']:,.0f}",
            f"AUD {cis['female_medium_salary_aud']:,.0f}",
            f"{cis['gender_pay_gap'] * 100:.1f}%",
        ],
    }
)

cis_kpis

## 4. Salary Ranking by Study Area

In [ ]:
salary_rank = area.sort_values("medium_term_median_salary_aud", ascending=False).dropna(
    subset=["medium_term_median_salary_aud"]
)

colors = ["#2f6f73" if value == "Yes" else "#c3a15f" for value in salary_rank["is_cis"]]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(salary_rank["field_of_education"], salary_rank["medium_term_median_salary_aud"], color=colors)
ax.invert_yaxis()
ax.set_title("Medium-term Median Salary by Study Area")
ax.set_xlabel("Median salary (AUD)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 5. Employment and Salary Relationship

In [ ]:
def point_colour(row):
    if row["is_cis"] == "Yes":
        return "#c4513b"
    if row["is_stem_related"] == "Yes":
        return "#2f6f73"
    return "#8aa6a3"


plot_data = area.dropna(subset=["medium_term_full_time_employment_rate", "medium_term_median_salary_aud"])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    plot_data["medium_term_full_time_employment_rate"],
    plot_data["medium_term_median_salary_aud"],
    c=plot_data.apply(point_colour, axis=1),
    s=70,
)
ax.scatter(
    [cis["medium_term_full_time_employment_rate"]],
    [cis["medium_term_median_salary_aud"]],
    c="#c4513b",
    s=120,
    label="CIS",
)
ax.annotate(
    "CIS",
    (cis["medium_term_full_time_employment_rate"], cis["medium_term_median_salary_aud"]),
    xytext=(8, 5),
    textcoords="offset points",
    color="#c4513b",
    weight="bold",
)
ax.set_title("Employment Rate and Salary by Study Area")
ax.set_xlabel("Medium-term full-time employment rate (%)")
ax.set_ylabel("Medium-term median salary (AUD)")
plt.tight_layout()
plt.show()

## 6. Gender Pay Gap

In [ ]:
gap_rank = area.dropna(subset=["gender_pay_gap"]).sort_values("gender_pay_gap", ascending=False)
gap_rank[["field_of_education", "male_medium_salary_aud", "female_medium_salary_aud", "gender_pay_gap"]].head(10)

In [ ]:
stem_gap = area.loc[area["is_stem_related"] == "Yes"].dropna(subset=["gender_pay_gap"])
stem_gap = stem_gap.sort_values("gender_pay_gap")

colors = ["#2f6f73" if value == "Yes" else "#b7a77d" for value in stem_gap["is_cis"]]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(stem_gap["field_of_education"], stem_gap["gender_pay_gap"] * 100, color=colors)
ax.set_title("Gender Pay Gap in STEM-related Study Areas")
ax.set_xlabel("Gender pay gap (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 7. University Outcomes

In [ ]:
institutions.sort_values("medium_term_median_salary_aud", ascending=False).head(10)[
    ["institution", "medium_term_median_salary_aud", "medium_term_full_time_employment_rate"]
]

## 8. Correlation Analysis

In [ ]:
numeric_cols = [
    "medium_term_full_time_employment_rate",
    "medium_term_overall_employment_rate",
    "short_term_median_salary_aud",
    "medium_term_median_salary_aud",
    "salary_growth",
    "gender_pay_gap",
]

area[numeric_cols].corr().round(3)

## 9. Summary Insights

- CIS graduates show strong medium-term employment outcomes, with 91.0% full-time employment and 92.0% overall employment.
- CIS medium-term median salary reaches AUD 100,000, with strong salary growth from short-term to medium-term.
- A gender pay gap remains visible in CIS, with male median salary higher than female median salary.
- The public QILT workbook is aggregated data, so the results are suitable for benchmarking and descriptive analytics rather than individual-level causal claims.